# Project 2 - Phase 3: Customer Lifetime Value (CLV) Modeling
**Customer Segmentation and Lifetime Value (CLV) Analysis**

This notebook covers **Week 3** of the project plan:
- Calculate key CLV inputs: Average Purchase Value, Purchase Frequency, and Customer Lifespan
- Implement a historical CLV formula
- (Stretch goal) Build a simple probabilistic CLV model using the BG/NBD approach

This continues from Phase 2, where the RFM table and customer clusters were created.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load the Data

We reload the cleaned transaction data and rebuild the same RFM + cluster table produced in Phase 2, so this notebook can run independently while staying consistent with the earlier phase.

In [ ]:
df = pd.read_csv("OnlineRetail.csv", encoding="latin1")

# Same cleaning steps as Phase 1 / Phase 2
df = df.dropna(subset=['CustomerID'])
df = df.drop_duplicates()
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

df.shape

In [ ]:
# Rebuild RFM table (same as Phase 2)
reference_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,
    "InvoiceNo": "nunique",
    "TotalAmount": "sum"
})

rfm.columns = ["Recency", "Frequency", "Monetary"]
rfm.head()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

rfm_scaled = StandardScaler().fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm.head()

## 2. Key Inputs for CLV

To calculate a historical CLV we need, for every customer:

- **Average Purchase Value** = Total spend / Number of transactions
- **Purchase Frequency** = Number of transactions (over the observed period)
- **Customer Lifespan** = How long (in days/months/years) the customer has been active, from their first to their last purchase

We compute customer lifespan directly from the raw transaction data (first vs. last purchase date), since the RFM table alone only stores Recency (days since last purchase).

In [ ]:
# First and last purchase date per customer -> lifespan
customer_dates = df.groupby("CustomerID")["InvoiceDate"].agg(["min", "max"])
customer_dates.columns = ["FirstPurchase", "LastPurchase"]

# Lifespan in days; customers with a single purchase get a minimum of 1 day
customer_dates["Lifespan_Days"] = (customer_dates["LastPurchase"] - customer_dates["FirstPurchase"]).dt.days
customer_dates["Lifespan_Days"] = customer_dates["Lifespan_Days"].replace(0, 1)

customer_dates.head()

In [ ]:
# Merge lifespan into the RFM/cluster table
clv_df = rfm.merge(customer_dates, left_index=True, right_index=True)

# Average Purchase Value = Monetary (total spend) / Frequency (number of transactions)
clv_df["AvgPurchaseValue"] = clv_df["Monetary"] / clv_df["Frequency"]

# Purchase Frequency = number of transactions
clv_df["PurchaseFrequency"] = clv_df["Frequency"]

# Express lifespan in months for a more intuitive/business-friendly CLV
clv_df["Lifespan_Months"] = clv_df["Lifespan_Days"] / 30.44

clv_df[["Recency", "Frequency", "Monetary", "AvgPurchaseValue",
        "PurchaseFrequency", "Lifespan_Days", "Lifespan_Months", "Cluster"]].head()

## 3. Historical CLV Formula

We use the standard historical CLV formula from the project brief:

**CLV = (Average Transaction Value × Number of Transactions) × Average Customer Lifespan**

Here "Average Transaction Value × Number of Transactions" is simply the customer's total historical spend (Monetary), so this scales total spend by the customer's lifespan span. Since most one-time buyers have a very short observed lifespan (close to 0), we use the lifespan in months so the metric stays meaningful and comparable across customers.

In [ ]:
clv_df["CLV"] = (clv_df["AvgPurchaseValue"] * clv_df["PurchaseFrequency"]) * clv_df["Lifespan_Months"]

clv_df[["Monetary", "AvgPurchaseValue", "PurchaseFrequency", "Lifespan_Months", "CLV"]].sort_values("CLV", ascending=False).head(10)

In [ ]:
clv_df["CLV"].describe()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(clv_df["CLV"], bins=50)
plt.title("Distribution of Historical CLV")
plt.xlabel("CLV")
plt.ylabel("Number of Customers")
plt.show()

> **Note:** The distribution is heavily right-skewed — a small number of high-value, long-tenured customers pull the average up. This is expected in retail/fintech customer bases and is exactly why segment-level analysis (Phase 4) matters more than a single average.

In [ ]:
# Save the CLV table for use in Phase 4 (segment insights & recommendations)
clv_df.to_csv("Customer_CLV.csv", index=True)
clv_df.shape

## 4. Stretch Goal: Probabilistic CLV with BG/NBD

The historical formula above only tells us what customers were worth *in the past*. A probabilistic model like **BG/NBD (Beta-Geometric/Negative Binomial Distribution)**, combined with a **Gamma-Gamma** model for spend, predicts *future* transactions and value — which is more useful for forward-looking business decisions.

This requires the `lifetimes` library:
```
pip install lifetimes
```

The model needs data in "summary" format: Frequency, Recency, T (customer age/duration since first purchase), and Monetary value — which maps directly onto our RFM table.

In [ ]:
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

# Build the lifetimes-required summary table directly from transactions
summary = summary_data_from_transaction_data(
    df,
    customer_id_col="CustomerID",
    datetime_col="InvoiceDate",
    monetary_value_col="TotalAmount",
    observation_period_end=df["InvoiceDate"].max()
)

summary.head()

In [ ]:
# Fit the BG/NBD model on frequency, recency, and T (customer age)
bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(summary["frequency"], summary["recency"], summary["T"])

# Predict expected number of purchases in the next 90 days
summary["predicted_purchases_90d"] = bgf.conditional_expected_number_of_purchases_up_to_time(
    90, summary["frequency"], summary["recency"], summary["T"]
)

summary.head()

In [ ]:
# Gamma-Gamma model needs customers with at least 1 repeat purchase and positive monetary value
returning_customers = summary[(summary["frequency"] > 0) & (summary["monetary_value"] > 0)]

ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(returning_customers["frequency"], returning_customers["monetary_value"])

# Predict CLV over the next 6 months (time is in months, discount_rate is a monthly rate)
summary["predicted_clv_6m"] = np.nan
summary.loc[returning_customers.index, "predicted_clv_6m"] = ggf.customer_lifetime_value(
    bgf,
    returning_customers["frequency"],
    returning_customers["recency"],
    returning_customers["T"],
    returning_customers["monetary_value"],
    time=6,          # months
    freq="D",
    discount_rate=0.01
)

summary.sort_values("predicted_clv_6m", ascending=False).head(10)

> The BG/NBD + Gamma-Gamma output only produces a predicted value for **returning customers** (frequency > 0), since one-time buyers don't give the model enough information to estimate repeat-purchase behaviour. One-time buyers are kept in the historical CLV metric instead.

## 5. Summary

- Built the three core CLV inputs (Average Purchase Value, Purchase Frequency, Customer Lifespan) for every customer.
- Calculated a **historical CLV** for each customer and saved it to `Customer_CLV.csv` for use in Phase 4.
- As a stretch goal, fit a **BG/NBD + Gamma-Gamma** probabilistic model to estimate forward-looking CLV for repeat customers.

**Next (Phase 4):** merge `Customer_CLV.csv` with the cluster labels to compute average CLV per segment, and turn the segments + CLV into business recommendations.